# Retail Data Analysis Report

This notebook loads processed purchase data and KPIs to generate visualizations.

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data and KPIs

In [ ]:
# Define file paths (relative to the notebooks directory)
processed_data_path = "../data/processed_purchases.csv"
kpis_path = "../data/kpis.json"

# Load processed data
try:
    df = pd.read_csv(processed_data_path)
    # Convert timestamp back to datetime if it's not already (CSV doesn't store type perfectly)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    print("Processed data loaded successfully.")
    print(f"Shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: Processed data file not found at {processed_data_path}")
    df = None

# Load KPIs
try:
    with open(kpis_path, 'r') as f:
        kpis = json.load(f)
    print("\nKPIs loaded successfully.")
    print(json.dumps(kpis, indent=2))
except FileNotFoundError:
    print(f"Error: KPIs file not found at {kpis_path}")
    kpis = None

## 2. Visualize KPIs

### 2.1 Total Revenue by Product Category

In [ ]:
if kpis and 'revenue_by_category' in kpis:
    revenue_by_cat = pd.Series(kpis['revenue_by_category'])
    plt.figure(figsize=(8, 5))
    sns.barplot(x=revenue_by_cat.index, y=revenue_by_cat.values, palette="viridis")
    plt.title('Total Revenue by Product Category')
    plt.xlabel('Product Category')
    plt.ylabel('Total Revenue ($)')
    plt.show()
else:
    print("KPI 'revenue_by_category' not available for plotting.")

### 2.2 Top N Selling Products (by Revenue)

In [ ]:
if kpis and 'top_selling_products' in kpis:
    top_products = pd.Series(kpis['top_selling_products'])
    plt.figure(figsize=(12, 7))
    sns.barplot(x=top_products.values, y=top_products.index, palette="mako", orient='h')
    plt.title(f'Top {len(top_products)} Selling Products by Revenue')
    plt.xlabel('Total Revenue ($)')
    plt.ylabel('Product Name')
    plt.tight_layout()
    plt.show()
else:
    print("KPI 'top_selling_products' not available for plotting.")

### 2.3 Preferred Payment Methods

In [ ]:
if kpis and 'preferred_payment_methods' in kpis:
    payment_methods = pd.Series(kpis['preferred_payment_methods'])
    plt.figure(figsize=(8, 8))
    plt.pie(payment_methods, labels=payment_methods.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
    plt.title('Preferred Payment Methods (by Transaction Count)')
    plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
    plt.show()
else:
    print("KPI 'preferred_payment_methods' not available for plotting.")

### 2.4 Revenue Over Time (Monthly)

In [ ]:
if df is not None and 'timestamp' in df.columns and 'total_purchase_amount' in df.columns:
    # Ensure timestamp is datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Resample to monthly frequency
    monthly_revenue = df.set_index('timestamp').resample('M')['total_purchase_amount'].sum()
    
    plt.figure(figsize=(12, 6))
    monthly_revenue.plot(kind='line', marker='o', linestyle='-')
    plt.title('Total Revenue Over Time (Monthly)')
    plt.xlabel('Month')
    plt.ylabel('Total Revenue ($)')
    plt.grid(True)
    plt.show()
else:
    print("DataFrame with 'timestamp' and 'total_purchase_amount' not available for plotting revenue over time.")